# eeg2img on Colab — training

Launches `scripts/train.py`; all logic lives in the repository. Run
`colab_prepare_data.ipynb` once first: this notebook reads the store it saved to Drive.

Runtime → Change runtime type → **T4 GPU**. The `GH_TOKEN` secret must be set as before.
Runs are resumable: if the session dies, re-run the same cell and training continues
from the last checkpoint on Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO = "moh246/eeg2image-neurips26"
DRIVE_DIR = "/content/drive/MyDrive/eeg2img"
STORE = "/content/stores/alljoined16m_v1"              # local copy: Drive is too slow to train from

import os, shutil, subprocess
if not os.path.isdir("/content/eeg2image-neurips26"):
    token = userdata.get("GH_TOKEN")
    subprocess.run(["git", "clone", "--quiet", f"https://{token}@github.com/{REPO}.git",
                    "/content/eeg2image-neurips26"], check=True)
if not os.path.isdir(STORE):
    shutil.copytree(f"{DRIVE_DIR}/stores/alljoined16m_v1", STORE)
%cd /content/eeg2image-neurips26
!git pull --quiet && git log -1 --format='%h %s'
%pip install --quiet -e ".[data]"

## Train one experiment

Pick a config from `configs/experiment/`. Records go to `experiments/` (committed to git
afterwards), checkpoints and the submission ZIP to Drive.

In [ ]:
EXPERIMENT = "alljoined_conv"      # alljoined_ridge | alljoined_eegnet | alljoined_conv
SEED = 0

!python scripts/train.py --config configs/experiment/{EXPERIMENT}.yaml seed={SEED} \
    data.store={STORE} data.embeddings={STORE}/embeddings.npy \
    train.checkpoint_dir={DRIVE_DIR}/checkpoints/{EXPERIMENT}_seed{SEED} \
    output_root={DRIVE_DIR}/outputs

## Save the run records

Copies the small run records to Drive and prints the registry line to paste back, so the
result can be committed with its config and commit hash.

In [ ]:
!mkdir -p {DRIVE_DIR}/experiments && cp -r experiments/runs experiments/registry.jsonl {DRIVE_DIR}/experiments/
!tail -n 1 experiments/registry.jsonl